# 🚕 Urban Cab Fare — EDA & Predictive Modeling
### End-to-end analysis by a Senior Data Scientist workflow

**Objective:** Understand what drives cab fares and surge pricing in an urban ride-hailing dataset, then build two predictive models:

1. **Regression** → Predict `Final_Fare` (₹) from trip/demand features
2. **Classification** → Predict `High_Surge` (whether a ride gets surge pricing ≥ 1.5×) from operational conditions

**Dataset:** `UrbanCabFare.csv` — 1,000 rides across 6 Indian cities with driver, demand-supply, traffic and fare fields.

**Workflow:**
1. Setup & Data Loading
2. Data Understanding & Quality Checks
3. Data Cleaning & Feature Engineering
4. Exploratory Data Analysis (univariate, bivariate, temporal)
5. Preprocessing Pipeline
6. Regression Modeling (Linear Regression vs Random Forest)
7. Classification Modeling (Logistic Regression vs Random Forest)
8. Feature Importance & Business Insights
9. Conclusions & Recommendations


## 1. Setup & Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & modeling
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Display / plotting settings
pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
print("Libraries loaded successfully.")


## 2. Load the Data

Run the cell below in Google Colab — it opens a file picker. Select `UrbanCabFare.csv`.
(If you've mounted Google Drive instead, just replace the path in the fallback line.)

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
except ImportError:
    # Not running in Colab — edit this path to point at your local copy
    file_name = "UrbanCabFare.csv"

df = pd.read_csv(file_name)
print(f"Loaded shape: {df.shape}")
df.head()


## 3. Data Understanding & Quality Checks

In [ ]:
df.info()


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print()
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])


In [ ]:
df.describe(include="number").T


In [ ]:
df.describe(include="object").T


**Observations so far:**
- `Driver_Name` has a small number of missing values — likely unregistered/guest drivers, not something to drop rows over.
- Numeric ranges look sane (no negative fares/distances), but we'll check for outliers formally below.
- `Ride_Date` is a string (`DD-MM-YYYY`) and needs parsing into a real datetime before we can extract time features.
- `User_ID` and `Driver_Name` are identifiers, not predictive features — they'll be excluded from modeling.

## 4. Data Cleaning & Feature Engineering

In [ ]:
df_clean = df.copy()

# --- Handle missing driver names (unknown/guest driver) ---
df_clean["Driver_Name"] = df_clean["Driver_Name"].fillna("Unknown")

# --- Parse date & extract time-based features ---
df_clean["Ride_Date"] = pd.to_datetime(df_clean["Ride_Date"], format="%d-%m-%Y")
df_clean["Month"] = df_clean["Ride_Date"].dt.month
df_clean["Is_Weekend"] = df_clean["Day_of_Week"].isin(["Saturday", "Sunday"]).astype(int)

# --- Time-of-day bucket from Hour_of_Day ---
def time_bucket(h):
    if 5 <= h < 12:
        return "Morning"
    elif 12 <= h < 17:
        return "Afternoon"
    elif 17 <= h < 21:
        return "Evening"
    else:
        return "Night"

df_clean["Time_of_Day"] = df_clean["Hour_of_Day"].apply(time_bucket)

# --- Engineered ratio features ---
df_clean["Fare_per_km"] = df_clean["Final_Fare"] / df_clean["Distance_km"]
df_clean["Fare_per_min"] = df_clean["Final_Fare"] / df_clean["Trip_Duration"]
df_clean["Surge_Amount"] = df_clean["Final_Fare"] - df_clean["Base_Fare"]

# --- Classification target: High_Surge (surge multiplier >= 1.5) ---
df_clean["High_Surge"] = (df_clean["Surge_Multiplier"] >= 1.5).astype(int)

print("Class balance for High_Surge target:")
print(df_clean["High_Surge"].value_counts(normalize=True).round(3))
df_clean.head()


In [ ]:
# Outlier check via IQR on key numeric columns
num_cols_check = ["Distance_km", "Trip_Duration", "Final_Fare", "Demand_Supply_Ratio"]
for col in num_cols_check:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    print(f"{col:22s} -> outliers: {n_out:3d}  (bounds: {lower:.2f} to {upper:.2f})")


`Demand_Supply_Ratio` shows the most outliers (right-skewed — a handful of rides have very high demand relative to supply). We keep these rows since they're genuine surge-driving events, not data errors, but we'll watch for their influence on linear models.

## 5. Exploratory Data Analysis

### 5.1 Target Variable Distribution — `Final_Fare`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df_clean["Final_Fare"], kde=True, bins=30, color="teal", ax=axes[0])
axes[0].set_title("Distribution of Final Fare")
axes[0].set_xlabel("Final Fare (₹)")

sns.boxplot(x=df_clean["Final_Fare"], color="orange", ax=axes[1])
axes[1].set_title("Final Fare — Boxplot")
axes[1].set_xlabel("Final Fare (₹)")

plt.tight_layout()
plt.show()

print(df_clean["Final_Fare"].describe().round(2))


### 5.2 Numeric Feature Distributions

In [ ]:
num_features = ["Distance_km", "Trip_Duration", "Base_Fare", "Surge_Multiplier",
                "Demand_Supply_Ratio", "No_of_active_drivers", "Ride_Requests"]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, col in enumerate(num_features):
    sns.histplot(df_clean[col], kde=True, bins=25, ax=axes[i], color="steelblue")
    axes[i].set_title(col)
for j in range(len(num_features), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


### 5.3 Categorical Feature Counts

In [ ]:
cat_features = ["City", "Traffic_Level", "Type_of_vehicle", "Day_of_Week", "Time_of_Day"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(cat_features):
    order = df_clean[col].value_counts().index
    sns.countplot(data=df_clean, x=col, order=order, ax=axes[i], palette="viridis")
    axes[i].set_title(f"Ride Count by {col}")
    axes[i].tick_params(axis="x", rotation=30)
for j in range(len(cat_features), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


### 5.4 What Drives Fare? — Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df_clean, x="Distance_km", y="Final_Fare",
                 hue="Type_of_vehicle", alpha=0.6, ax=axes[0])
axes[0].set_title("Final Fare vs Distance (by Vehicle Type)")

sns.scatterplot(data=df_clean, x="Trip_Duration", y="Final_Fare",
                 hue="Traffic_Level", alpha=0.6, ax=axes[1])
axes[1].set_title("Final Fare vs Trip Duration (by Traffic Level)")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x="City", y="Final_Fare", ax=axes[0], palette="Set2")
axes[0].set_title("Final Fare by City")
axes[0].tick_params(axis="x", rotation=30)

sns.boxplot(data=df_clean, x="Type_of_vehicle", y="Final_Fare", ax=axes[1], palette="Set3")
axes[1].set_title("Final Fare by Vehicle Type")

plt.tight_layout()
plt.show()


### 5.5 What Drives Surge Pricing?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x="Traffic_Level", y="Surge_Multiplier",
            order=["Low", "Medium", "High", "Jam"], ax=axes[0], palette="rocket")
axes[0].set_title("Surge Multiplier by Traffic Level")

sns.scatterplot(data=df_clean, x="Demand_Supply_Ratio", y="Surge_Multiplier",
                 alpha=0.5, ax=axes[1], color="crimson")
axes[1].set_title("Surge Multiplier vs Demand/Supply Ratio")
axes[1].set_xlim(0, 20)

plt.tight_layout()
plt.show()


### 5.6 Temporal Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly = df_clean.groupby("Hour_of_Day")["Final_Fare"].mean()
sns.lineplot(x=hourly.index, y=hourly.values, marker="o", ax=axes[0], color="darkgreen")
axes[0].set_title("Average Fare by Hour of Day")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Avg Final Fare (₹)")

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily = df_clean.groupby("Day_of_Week")["Final_Fare"].mean().reindex(day_order)
sns.barplot(x=daily.index, y=daily.values, ax=axes[1], palette="coolwarm")
axes[1].set_title("Average Fare by Day of Week")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


### 5.7 Correlation Heatmap

In [ ]:
corr_cols = ["Hour_of_Day", "Distance_km", "No_of_active_drivers", "Ride_Requests",
             "Demand_Supply_Ratio", "Trip_Duration", "Base_Fare", "Surge_Multiplier", "Final_Fare"]

plt.figure(figsize=(10, 7))
sns.heatmap(df_clean[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix — Numeric Features")
plt.tight_layout()
plt.show()


In [ ]:
# Does surge follow a near-deterministic rule based on demand/supply?
bins = [0, 1, 2, 3, 5, df_clean["Demand_Supply_Ratio"].max()]
surge_by_ratio = df_clean.groupby(pd.cut(df_clean["Demand_Supply_Ratio"], bins=bins))["Surge_Multiplier"] \
    .agg(["mean", "min", "max", "count"])
print(surge_by_ratio)
print(f"\nCorrelation(Demand_Supply_Ratio, Surge_Multiplier) = "
      f"{df_clean['Demand_Supply_Ratio'].corr(df_clean['Surge_Multiplier']):.3f}")


Notice the surge multiplier is almost fully determined by which demand/supply bucket a ride falls into (ratio ≤ 1 → 1.0×, ratio > 2 → ≥ 1.5×). This is a strong, near-rule-based signal — worth remembering when we get suspiciously high classification scores later.

**Key EDA takeaways:**
- `Final_Fare` is strongly driven by `Distance_km` and `Base_Fare` (near-linear relationship), with `Surge_Multiplier` adding a secondary, non-linear boost.
- `Surge_Multiplier` correlates only weakly with raw traffic level but rises with `Demand_Supply_Ratio` — real scarcity of drivers matters more than congestion labels.
- Fare and surge both show mild time-of-day patterns (peaks around commute hours), useful signal for the models.
- City and vehicle type shift the fare *level* but not its relationship with distance — consistent with different base rate cards per city/vehicle.

## 6. Preprocessing Pipeline

We build one shared preprocessing strategy (`ColumnTransformer`) — numeric features are scaled, categorical features are one-hot encoded — and reuse it for both tasks via `sklearn` Pipelines. This keeps the workflow leak-free and reproducible.

In [ ]:
# Feature set shared by both models (raw identifiers & date excluded)
feature_cols = [
    "City", "Hour_of_Day", "Day_of_Week", "Time_of_Day", "Is_Weekend",
    "Distance_km", "Traffic_Level", "Type_of_vehicle",
    "No_of_active_drivers", "Ride_Requests", "Demand_Supply_Ratio", "Trip_Duration"
]

categorical_cols = ["City", "Day_of_Week", "Time_of_Day", "Traffic_Level", "Type_of_vehicle"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_cols)
])

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)


## 7. Regression Modeling — Predicting `Final_Fare`

**Note:** `Base_Fare` and `Surge_Amount` are deliberately excluded from the regression feature set — they are near-deterministic components of `Final_Fare` itself (`Final_Fare ≈ Base_Fare × Surge_Multiplier`), so including them would leak the answer rather than test genuine predictive power from operating conditions.

In [ ]:
X = df_clean[feature_cols]
y_reg = df_clean["Final_Fare"]

X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# --- Model A: Linear Regression (interpretable baseline) ---
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
lr_pipeline.fit(X_train, y_train_reg)
lr_preds = lr_pipeline.predict(X_test)

# --- Model B: Random Forest Regressor (captures non-linearity/interactions) ---
rf_reg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=300, max_depth=10, random_state=RANDOM_STATE))
])
rf_reg_pipeline.fit(X_train, y_train_reg)
rf_reg_preds = rf_reg_pipeline.predict(X_test)

def regression_report(name, y_true, y_pred):
    return {
        "Model": name,
        "MAE": round(mean_absolute_error(y_true, y_pred), 2),
        "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        "R2": round(r2_score(y_true, y_pred), 4)
    }

reg_results = pd.DataFrame([
    regression_report("Linear Regression", y_test_reg, lr_preds),
    regression_report("Random Forest Regressor", y_test_reg, rf_reg_preds)
])
reg_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, preds, name in zip(axes, [lr_preds, rf_reg_preds], ["Linear Regression", "Random Forest"]):
    ax.scatter(y_test_reg, preds, alpha=0.5, color="teal")
    lims = [min(y_test_reg.min(), preds.min()), max(y_test_reg.max(), preds.max())]
    ax.plot(lims, lims, "r--", label="Perfect prediction")
    ax.set_xlabel("Actual Final Fare (₹)")
    ax.set_ylabel("Predicted Final Fare (₹)")
    ax.set_title(f"{name}: Actual vs Predicted")
    ax.legend()

plt.tight_layout()
plt.show()


### 7.1 Feature Importance (Random Forest Regressor)

In [ ]:
feature_names = (numeric_cols +
    list(rf_reg_pipeline.named_steps["preprocessor"]
         .named_transformers_["cat"].get_feature_names_out(categorical_cols)))

importances = rf_reg_pipeline.named_steps["model"].feature_importances_
fi_reg = pd.DataFrame({"feature": feature_names, "importance": importances}) \
            .sort_values("importance", ascending=False).head(12)

plt.figure(figsize=(9, 6))
sns.barplot(data=fi_reg, x="importance", y="feature", palette="mako")
plt.title("Top Features Driving Final Fare (Random Forest)")
plt.tight_layout()
plt.show()


## 8. Classification Modeling — Predicting `High_Surge`

Target: `High_Surge = 1` when `Surge_Multiplier >= 1.5` (roughly balanced at ~46% positive class), else `0`. This tells us, from *operating conditions alone*, whether a ride is likely to hit heavy surge pricing — useful for driver positioning and demand-side alerts.

In [ ]:
y_clf = df_clean["High_Surge"]

X_train_c, X_test_c, y_train_clf, y_test_clf = train_test_split(
    X, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)
print("Train class balance:\n", y_train_clf.value_counts(normalize=True).round(3))


In [ ]:
# --- Model A: Logistic Regression (interpretable baseline) ---
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
logreg_pipeline.fit(X_train_c, y_train_clf)
logreg_preds = logreg_pipeline.predict(X_test_c)
logreg_proba = logreg_pipeline.predict_proba(X_test_c)[:, 1]

# --- Model B: Random Forest Classifier ---
rf_clf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE))
])
rf_clf_pipeline.fit(X_train_c, y_train_clf)
rf_clf_preds = rf_clf_pipeline.predict(X_test_c)
rf_clf_proba = rf_clf_pipeline.predict_proba(X_test_c)[:, 1]

def classification_report_row(name, y_true, y_pred, y_proba):
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 3),
        "Precision": round(precision_score(y_true, y_pred), 3),
        "Recall": round(recall_score(y_true, y_pred), 3),
        "F1": round(f1_score(y_true, y_pred), 3),
        "ROC-AUC": round(roc_auc_score(y_true, y_proba), 3)
    }

clf_results = pd.DataFrame([
    classification_report_row("Logistic Regression", y_test_clf, logreg_preds, logreg_proba),
    classification_report_row("Random Forest Classifier", y_test_clf, rf_clf_preds, rf_clf_proba)
])
clf_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, preds, name in zip(axes, [logreg_preds, rf_clf_preds], ["Logistic Regression", "Random Forest"]):
    cm = confusion_matrix(y_test_clf, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Surge", "High Surge"], yticklabels=["No Surge", "High Surge"])
    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

print("=== Logistic Regression Report ===")
print(classification_report(y_test_clf, logreg_preds, target_names=["No Surge", "High Surge"]))
print("=== Random Forest Report ===")
print(classification_report(y_test_clf, rf_clf_preds, target_names=["No Surge", "High Surge"]))


In [ ]:
plt.figure(figsize=(7, 6))
for preds_proba, name, color in [(logreg_proba, "Logistic Regression", "darkorange"),
                                   (rf_clf_proba, "Random Forest", "teal")]:
    fpr, tpr, _ = roc_curve(y_test_clf, preds_proba)
    auc = roc_auc_score(y_test_clf, preds_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", color=color)

plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — High Surge Classification")
plt.legend()
plt.tight_layout()
plt.show()


### 8.1 Feature Importance (Random Forest Classifier)

In [ ]:
importances_clf = rf_clf_pipeline.named_steps["model"].feature_importances_
fi_clf = pd.DataFrame({"feature": feature_names, "importance": importances_clf}) \
            .sort_values("importance", ascending=False).head(12)

plt.figure(figsize=(9, 6))
sns.barplot(data=fi_clf, x="importance", y="feature", palette="flare")
plt.title("Top Features Driving High Surge Prediction (Random Forest)")
plt.tight_layout()
plt.show()


## 9. Model Comparison Summary

In [ ]:
print("REGRESSION — Predicting Final_Fare")
display(reg_results)

print("\nCLASSIFICATION — Predicting High_Surge")
display(clf_results)


## 10. Business Insights & Recommendations

**On fares:**
- `Distance_km` is, unsurprisingly, the dominant driver of `Final_Fare`; the Random Forest regressor captures the modest non-linear surge effect that plain linear regression underfits, giving it the edge on RMSE/R².
- City and vehicle type act mainly as level shifters — useful for setting per-city/vehicle base rates, less useful for predicting *within-segment* variation.

**On surge pricing:**
- `Demand_Supply_Ratio` and `Ride_Requests` are the strongest predictors of high surge — far more informative than the coarse `Traffic_Level` label. This suggests the platform's surge algorithm (and any forecasting model built on top of it) should weight live demand/supply signals over traffic congestion data.
- Both classifiers perform far above the ~54/46 baseline, with near-perfect scores. **Caveat:** cross-tabulating `Demand_Supply_Ratio` against `Surge_Multiplier` shows the relationship is almost a hard rule in this dataset (ratio ≤ 1 → surge locked at 1.0×; ratio > 2 → surge ≥ 1.5×), which is why accuracy is so high. In a real production dataset, this relationship is usually noisier — treat these scores as an upper bound and expect materially lower (though still strong) performance on live data.

**Recommendations:**
1. Use the regression model to give riders more accurate fare estimates upfront, especially for longer trips where surge compounding matters most.
2. Feed the classification model's surge probability into driver-facing apps to proactively reposition drivers toward high-demand zones *before* surge hits, rather than reacting to it.
3. Collect a finer-grained real-time demand/supply feed (beyond the current 4-level `Traffic_Level` bucket) — it's clearly the highest-leverage signal and worth the investment to sharpen.
4. Consider monitoring `Fare_per_km` and `Fare_per_min` as pricing-health KPIs to detect anomalous or unfair pricing at the trip level over time.

---
*Notebook generated as a complete, reproducible Colab workflow — every cell can be re-run top to bottom on a fresh copy of `UrbanCabFare.csv`.*